# 🔀 Checkpoint Score Endpoint Tests

Tests the **`POST /checkpoint/score`** endpoint — a structured, LLM-free entry point
for site scoring that bypasses the conversational chat node.

### Prerequisites
Start the server first:
```bash
uvicorn main:app --reload --port 8000
```

### Execution Path
```
POST /checkpoint/score
  → chat_node (short-circuit, no LLM)
  → orchestrator (fast-path, intent = score_site)
  → fetch_features → fetch_scores → validation → compute_score
  → explainability → insight → chat_response (skip formatting)
  → END → API route serializes CheckpointScoreResponse
```

| # | Test | Description |
|---|------|-------------|
| 1 | With weights | Retail site in Ahmedabad, explicit weights |
| 2 | Without weights | EV charging in Mumbai, advisory defaults |
| 3 | Different use cases | Warehouse, hospital, hotel |
| 4 | Error: invalid use_case | Should return 422 |
| 5 | Error: missing fields | Should return 422 |
| 6 | Error: out of bounds | Coordinates outside India |
| 7 | Latency comparison | Checkpoint vs /score benchmark |

---
## 0. Setup

In [1]:
import httpx
import json
import time
from pprint import pprint

BASE_URL = "http://127.0.0.1:8000"

client = httpx.AsyncClient(base_url=BASE_URL, timeout=120.0)


def show(resp: httpx.Response, label: str = ""):
    """Pretty-print an HTTP response with checkpoint-aware summarization."""
    status_icon = "✅" if resp.status_code < 400 else "❌"
    print(f"\n{status_icon} [{resp.status_code}] {resp.request.method} {resp.request.url}")
    if label:
        print(f"   {label}")
    print(f"   Time: {resp.elapsed.total_seconds():.2f}s")
    try:
        data = resp.json()
        # Checkpoint-specific summary
        if "final_score" in data and "thread_id" in data:
            print(f"\n   📍 Location:  {data.get('location', 'N/A')}")
            print(f"   🏷️  Use Case:  {data.get('use_case', 'N/A')}")
            print(f"   📊 Score:     {data.get('final_score', 'N/A')}/100")
            print(f"   🆔 Site ID:   {data.get('site_id', 'N/A')}")
            print(f"   🧵 Thread:    {data.get('thread_id', 'N/A')}")
            # Score breakdown
            bd = data.get("score_breakdown")
            if bd:
                print(f"   ✅ Strengths:  {bd.get('strengths', [])}")
                print(f"   ⚠️  Weaknesses: {bd.get('weaknesses', [])}")
                print(f"   📐 Contributions:")
                for dim, c in bd.get("contributions", {}).items():
                    print(f"       {dim:25s} raw={c['raw']:5.1f}  w={c['weight']:.2f}  contrib={c['contribution']:5.1f}")
            # Weights used
            wt = data.get("weights_used")
            if wt:
                print(f"   ⚖️  Weights:   {json.dumps(wt, indent=None)}")
            # Advisory text
            if data.get("advisory_text"):
                adv = data["advisory_text"]
                print(f"   💡 Advisory:  {adv[:200]}{'...' if len(adv) > 200 else ''}")
            # Validation warnings
            warnings = data.get("validation_warnings", [])
            if warnings:
                print(f"   ⚠️  Warnings ({len(warnings)}):")
                for w in warnings:
                    print(f"       • {w}")
            # Insight excerpt
            if data.get("insight_text"):
                insight = data["insight_text"]
                print(f"\n   🧠 Insight: {insight[:300]}{'...' if len(insight) > 300 else ''}")
        else:
            # Generic JSON output
            raw = json.dumps(data, indent=2, default=str)
            if len(raw) > 3000:
                print(raw[:3000])
                print(f"\n   ... ({len(raw):,} chars total, truncated)")
            else:
                print(raw)
    except Exception:
        print(resp.text[:500])
    return resp


# Quick health check
resp = await client.get("/")
if resp.status_code == 200:
    print(f"✅ Server is running at {BASE_URL}")
else:
    print(f"❌ Server not reachable at {BASE_URL} — start it first!")

✅ Server is running at http://127.0.0.1:8000


---
## 1. ✅ Checkpoint Score — With Explicit Weights

Retail site in Ahmedabad with manually specified weights.  
This should bypass the advisory node entirely.

In [2]:
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    }
})
show(resp, "Checkpoint score — Ahmedabad (retail, with explicit weights)");


✅ [200] POST http://127.0.0.1:8000/checkpoint/score
   Checkpoint score — Ahmedabad (retail, with explicit weights)
   Time: 26.52s

   📍 Location:  Ahmadabad, Gujarat
   🏷️  Use Case:  retail
   📊 Score:     63.77/100
   🆔 Site ID:   IND_0099352
   🧵 Thread:    2386ddcb-96ee-47aa-9b6b-01d4350b0a71
   ✅ Strengths:  ['demand_score', 'accessibility_score']
   ⚠️  Weaknesses: ['competition_score', 'risk_score']
   📐 Contributions:
       demand_score              raw= 67.0  w=0.30  contrib= 20.1
       accessibility_score       raw=100.0  w=0.20  contrib= 20.0
       suitability_score         raw=100.0  w=0.10  contrib= 10.0
       infrastructure_score      raw= 86.4  w=0.10  contrib=  8.6
       risk_score                raw= 50.4  w=0.10  contrib=  5.0
       competition_score         raw=  0.0  w=0.20  contrib=  0.0
   ⚖️  Weights:   {"demand_score": 0.3, "accessibility_score": 0.2, "competition_score": 0.2, "suitability_score": 0.1, "risk_score": 0.1, "infrastructure_score": 0.1}
   

---
## 2. ✅ Checkpoint Score — Without Weights (Advisory Defaults)

EV charging site in Mumbai — no weights provided.  
The advisory node should recommend default weights.

In [3]:
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 19.0760, "lng": 72.8777},
    "use_case": "ev_charging"
})
show(resp, "Checkpoint score — Mumbai (EV charging, advisory weights)");


✅ [200] POST http://127.0.0.1:8000/checkpoint/score
   Checkpoint score — Mumbai (EV charging, advisory weights)
   Time: 1.77s

   📍 Location:  MumbaiSuburban, Maharashtra
   🏷️  Use Case:  ev_charging
   📊 Score:     80.01/100
   🆔 Site ID:   IND_0050492
   🧵 Thread:    e5ff9302-3af0-46c7-8c63-027c0b90fdab
   ✅ Strengths:  ['accessibility_score', 'demand_score']
   ⚠️  Weaknesses: ['competition_score', 'risk_score']
   📐 Contributions:
       accessibility_score       raw= 99.4  w=0.30  contrib= 29.8
       demand_score              raw= 68.4  w=0.25  contrib= 17.1
       infrastructure_score      raw= 83.3  w=0.20  contrib= 16.7
       suitability_score         raw=100.0  w=0.10  contrib= 10.0
       risk_score                raw= 64.4  w=0.10  contrib=  6.4
       competition_score         raw=  0.0  w=0.05  contrib=  0.0
   ⚖️  Weights:   {"demand_score": 0.25, "accessibility_score": 0.3, "competition_score": 0.05, "suitability_score": 0.1, "risk_score": 0.1, "infrastructure_scor

---
## 3. ✅ Checkpoint Score — Multiple Use Cases

Test with different use case keys to verify the catalog validator.

In [4]:
# Warehouse in Bangalore
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 12.9716, "lng": 77.5946},
    "use_case": "warehouse"
})
show(resp, "Checkpoint — Bangalore (warehouse)");


✅ [200] POST http://127.0.0.1:8000/checkpoint/score
   Checkpoint — Bangalore (warehouse)
   Time: 1.57s

   📍 Location:  Bangalore, Karnataka
   🏷️  Use Case:  warehouse
   📊 Score:     86.81/100
   🆔 Site ID:   IND_0012948
   🧵 Thread:    4f9215f3-c758-4a32-b97f-abe99fd1ba5d
   ✅ Strengths:  ['accessibility_score', 'suitability_score']
   ⚠️  Weaknesses: ['competition_score', 'demand_score']
   📐 Contributions:
       accessibility_score       raw=100.0  w=0.35  contrib= 35.0
       suitability_score         raw=100.0  w=0.15  contrib= 15.0
       infrastructure_score      raw= 95.9  w=0.15  contrib= 14.4
       risk_score                raw= 85.5  w=0.15  contrib= 12.8
       demand_score              raw= 64.0  w=0.15  contrib=  9.6
       competition_score         raw=  0.0  w=0.05  contrib=  0.0
   ⚖️  Weights:   {"demand_score": 0.15, "accessibility_score": 0.35, "competition_score": 0.05, "suitability_score": 0.15, "risk_score": 0.15, "infrastructure_score": 0.15}

   🧠 Insigh

In [5]:
# Hospital in Delhi
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 28.6139, "lng": 77.2090},
    "use_case": "hospital"
})
show(resp, "Checkpoint — Delhi (hospital)");


✅ [200] POST http://127.0.0.1:8000/checkpoint/score
   Checkpoint — Delhi (hospital)
   Time: 1.58s

   📍 Location:  West, NCTofDelhi
   🏷️  Use Case:  hospital
   📊 Score:     74.47/100
   🆔 Site ID:   IND_0187837
   🧵 Thread:    6c09f933-66ce-45d0-8212-b5a4aab02913
   ✅ Strengths:  ['accessibility_score', 'demand_score']
   ⚠️  Weaknesses: ['competition_score', 'infrastructure_score']
   📐 Contributions:
       accessibility_score       raw=100.0  w=0.25  contrib= 25.0
       demand_score              raw= 64.2  w=0.30  contrib= 19.3
       risk_score                raw= 77.5  w=0.15  contrib= 11.6
       suitability_score         raw=100.0  w=0.10  contrib= 10.0
       infrastructure_score      raw= 85.8  w=0.10  contrib=  8.6
       competition_score         raw=  0.0  w=0.10  contrib=  0.0
   ⚖️  Weights:   {"demand_score": 0.3, "accessibility_score": 0.25, "competition_score": 0.1, "suitability_score": 0.1, "risk_score": 0.15, "infrastructure_score": 0.1}

   🧠 Insight: The hosp

In [6]:
# Hotel in Jaipur
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 26.9124, "lng": 75.7873},
    "use_case": "hotel"
})
show(resp, "Checkpoint — Jaipur (hotel)");


✅ [200] POST http://127.0.0.1:8000/checkpoint/score
   Checkpoint — Jaipur (hotel)
   Time: 1.78s

   📍 Location:  Jaipur, Rajasthan
   🏷️  Use Case:  hotel
   📊 Score:     65.02/100
   🆔 Site ID:   IND_0165457
   🧵 Thread:    08751e2a-b482-4301-80cb-c70cb0b10e86
   ✅ Strengths:  ['accessibility_score', 'demand_score']
   ⚠️  Weaknesses: ['competition_score', 'risk_score']
   📐 Contributions:
       accessibility_score       raw= 99.9  w=0.25  contrib= 25.0
       demand_score              raw= 57.5  w=0.22  contrib= 12.7
       infrastructure_score      raw= 99.5  w=0.10  contrib=  9.9
       suitability_score         raw= 53.0  w=0.18  contrib=  9.6
       risk_score                raw= 79.0  w=0.10  contrib=  7.9
       competition_score         raw=  0.0  w=0.15  contrib=  0.0
   ⚖️  Weights:   {"demand_score": 0.22, "accessibility_score": 0.25, "competition_score": 0.15, "suitability_score": 0.18, "risk_score": 0.1, "infrastructure_score": 0.1}

   🧠 Insight: The hotel site in Ja

---
## 4. ❌ Error: Invalid Use Case

Should return **422** with a validation error listing valid use cases.

In [7]:
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "banana_farm"
})
show(resp, "Expected 422 — invalid use_case 'banana_farm'");


❌ [422] POST http://127.0.0.1:8000/checkpoint/score
   Expected 422 — invalid use_case 'banana_farm'
   Time: 0.01s
{
  "detail": [
    {
      "type": "value_error",
      "loc": [
        "body",
        "use_case"
      ],
      "msg": "Value error, Invalid use_case 'banana_farm'. Valid options: ['agri_cold_chain', 'agri_input_store', 'agri_mandi', 'amusement_park', 'anganwadi', 'aquaculture', 'art_gallery', 'atm', 'auto_ancillary', 'auto_service', 'ayurvedic_center', 'bakery', 'bank_branch', 'bar', 'blood_bank', 'bowling_alley', 'cafe', 'chemical_factory', 'clinic', 'cloud_kitchen', 'cng_station', 'cold_storage', 'college', 'community_hall', 'coworking', 'crematorium', 'data_center', 'delivery_hub', 'dhaba', 'dharamshala', 'diagnostic_lab', 'dialysis_center', 'etp', 'ev_charging', 'ev_swap', 'fertiliser_plant', 'fire_station', 'food_processing', 'gaming_zone', 'gas_bottling', 'gidc_plot', 'govt_office', 'greenhouse', 'gym', 'hospital', 'hotel', 'insurance_office', 'juice_bar', 'la

---
## 5. ❌ Error: Missing Required Fields

Test with missing `site_input` and missing `use_case`.

In [8]:
# Missing site_input entirely
resp = await client.post("/checkpoint/score", json={
    "use_case": "retail"
})
show(resp, "Expected 422 — missing site_input");


❌ [422] POST http://127.0.0.1:8000/checkpoint/score
   Expected 422 — missing site_input
   Time: 0.00s
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "site_input"
      ],
      "msg": "Field required",
      "input": {
        "use_case": "retail"
      }
    }
  ]
}


In [ ]:
# Missing use_case
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714}
})
show(resp, "Expected 422 — missing use_case");

In [ ]:
# Empty body
resp = await client.post("/checkpoint/score", json={})
show(resp, "Expected 422 — empty request body");

---
## 6. ❌ Error: Out-of-Bounds Coordinates

Coordinates outside India bounds should be rejected by the orchestrator's  
coordinate validation (even though checkpoint fast-paths to `score_site`,  
the fetch_features node will fail for non-existent locations).

In [ ]:
# New York City — far outside India
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 40.7128, "lng": -74.0060},
    "use_case": "retail"
})
show(resp, "Expected error — coordinates outside India (New York)");

---
## 7. ⚖️ Weights Sum Validation

Test with weights that don't sum to 1.0 — should error during graph execution.

In [ ]:
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.50,
        "accessibility_score": 0.50,
        "competition_score": 0.50,
        "suitability_score": 0.50,
        "risk_score": 0.50,
        "infrastructure_score": 0.50
    }
})
show(resp, "Expected error — weights sum to 3.0");

---
## 8. ⏱️ Latency Comparison: `/checkpoint/score` vs `/score`

The checkpoint endpoint should be **faster** because it skips the LLM  
intent detection call in `chat_node` and the orchestrator parsing logic.

In [ ]:
N_RUNS = 3

CHECKPOINT_PAYLOAD = {
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    }
}

SCORE_PAYLOAD = {
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    }
}

print("=" * 60)
print(f"📊 Latency Benchmark — {N_RUNS} runs each")
print("=" * 60)

# Benchmark /checkpoint/score
print(f"\n🔀 POST /checkpoint/score")
checkpoint_times = []
for i in range(N_RUNS):
    start = time.perf_counter()
    resp = await client.post("/checkpoint/score", json=CHECKPOINT_PAYLOAD)
    elapsed = time.perf_counter() - start
    checkpoint_times.append(elapsed)
    status = "✅" if resp.status_code == 200 else "❌"
    print(f"  Run {i+1}/{N_RUNS}: {elapsed:.3f}s {status}")

# Benchmark /score
print(f"\n📝 POST /score")
score_times = []
for i in range(N_RUNS):
    start = time.perf_counter()
    resp = await client.post("/score", json=SCORE_PAYLOAD)
    elapsed = time.perf_counter() - start
    score_times.append(elapsed)
    status = "✅" if resp.status_code == 200 else "❌"
    print(f"  Run {i+1}/{N_RUNS}: {elapsed:.3f}s {status}")

# Summary
print(f"\n{'─' * 60}")
print(f"📊 Summary")
print(f"{'─' * 60}")
print(f"{'Metric':<20} {'Checkpoint':>12} {'Score':>12} {'Δ':>12}")
print(f"{'─' * 60}")

cp_avg = sum(checkpoint_times) / len(checkpoint_times)
sc_avg = sum(score_times) / len(score_times)
cp_min = min(checkpoint_times)
sc_min = min(score_times)

print(f"{'Avg':.<20} {cp_avg:>11.3f}s {sc_avg:>11.3f}s {sc_avg - cp_avg:>+11.3f}s")
print(f"{'Min':.<20} {cp_min:>11.3f}s {sc_min:>11.3f}s {sc_min - cp_min:>+11.3f}s")
print(f"{'Max':.<20} {max(checkpoint_times):>11.3f}s {max(score_times):>11.3f}s {max(score_times) - max(checkpoint_times):>+11.3f}s")

if cp_avg < sc_avg:
    pct = ((sc_avg - cp_avg) / sc_avg) * 100
    print(f"\n🚀 Checkpoint is {pct:.1f}% faster on average")
else:
    print(f"\n⚠️  /score was faster — LLM overhead may be negligible for this config")

📊 Latency Benchmark — 3 runs each

🔀 POST /checkpoint/score
  Run 1/3: 2.088s ✅
  Run 2/3: 1.609s ✅
  Run 3/3: 1.863s ✅

📝 POST /score
  Run 1/3: 3.074s ❌
  Run 2/3: 3.601s ❌


---
## 9. 🔍 Response Schema Validation

Verify that the checkpoint response has all expected fields
and correct types.

In [ ]:
resp = await client.post("/checkpoint/score", json={
    "site_input": {"lat": 23.0225, "lng": 72.5714},
    "use_case": "retail",
    "weights": {
        "demand_score": 0.30,
        "accessibility_score": 0.20,
        "competition_score": 0.20,
        "suitability_score": 0.10,
        "risk_score": 0.10,
        "infrastructure_score": 0.10
    }
})

assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
data = resp.json()

# Check required fields exist
REQUIRED_FIELDS = [
    "site_id", "location", "use_case", "final_score",
    "score_breakdown", "weights_used", "insight_text",
    "validation_warnings", "thread_id",
]

print("🔍 Response Schema Validation")
print("=" * 50)
all_pass = True
for field in REQUIRED_FIELDS:
    present = field in data
    icon = "✅" if present else "❌"
    value_type = type(data.get(field)).__name__ if present else "MISSING"
    print(f"  {icon} {field:25s} type={value_type}")
    if not present:
        all_pass = False

# Type checks
print(f"\n📐 Type Assertions")
print("=" * 50)

checks = [
    ("site_id is str", isinstance(data.get("site_id"), str)),
    ("location is str", isinstance(data.get("location"), str)),
    ("use_case == 'retail'", data.get("use_case") == "retail"),
    ("final_score is float", isinstance(data.get("final_score"), (int, float))),
    ("final_score in [0, 100]", 0 <= (data.get("final_score") or 0) <= 100),
    ("score_breakdown is dict", isinstance(data.get("score_breakdown"), dict)),
    ("weights_used is dict", isinstance(data.get("weights_used"), dict)),
    ("validation_warnings is list", isinstance(data.get("validation_warnings"), list)),
    ("thread_id is str", isinstance(data.get("thread_id"), str)),
    ("insight_text is str", isinstance(data.get("insight_text"), str)),
]

for label, passed in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon} {label}")
    if not passed:
        all_pass = False

# Breakdown sub-fields
if data.get("score_breakdown"):
    bd = data["score_breakdown"]
    EXPECTED_DIMS = [
        "demand_score", "accessibility_score", "competition_score",
        "suitability_score", "risk_score", "infrastructure_score",
    ]
    print(f"\n📊 Score Breakdown Dimensions")
    print("=" * 50)
    for dim in EXPECTED_DIMS:
        present = dim in bd.get("contributions", {})
        icon = "✅" if present else "❌"
        if present:
            c = bd["contributions"][dim]
            print(f"  {icon} {dim:25s} raw={c['raw']:5.1f}  w={c['weight']:.2f}  contrib={c['contribution']:5.1f}")
        else:
            print(f"  {icon} {dim:25s} MISSING")
            all_pass = False

print(f"\n{'🎉 ALL CHECKS PASSED' if all_pass else '⚠️  SOME CHECKS FAILED'}")

---
## 10. 📋 Verify Endpoint in OpenAPI Spec

Confirm that `/checkpoint/score` is registered in the Swagger docs.

In [ ]:
resp = await client.get("/openapi.json")
if resp.status_code == 200:
    spec = resp.json()
    paths = spec.get("paths", {})
    
    checkpoint_path = "/checkpoint/score"
    if checkpoint_path in paths:
        entry = paths[checkpoint_path]
        print(f"✅ {checkpoint_path} is registered in OpenAPI spec")
        for method, details in entry.items():
            print(f"   Method:  {method.upper()}")
            print(f"   Summary: {details.get('summary', 'N/A')}")
            print(f"   Tags:    {details.get('tags', [])}")
    else:
        print(f"❌ {checkpoint_path} NOT found in OpenAPI spec")
        print(f"   Available paths:")
        for path in sorted(paths.keys()):
            print(f"     {path}")
else:
    print(f"❌ Failed to get OpenAPI spec: {resp.status_code}")

---
## 11. 🧹 Cleanup

In [ ]:
await client.aclose()
print("HTTP client closed ✓")